# 2. Find the interpreter

For every broadcast of `work/broadcasts.json`, while its download is on disk:

1. **Find** the frame ranges in which the interpreter is on screen.
   - **PrimaTV and ProTV** interpret a stretch of 25 to 30 minutes per broadcast, found by face
     recognition of the channel's interpreters (`known_people/`) in a frame every 30 s, with
     binary searches for where the interpreter appears and leaves.
   - **Digi24** scatters interpretation over the journal. A small classifier marks every frame
     with or without the interpreter, and a state machine turns that into ranges.

   Then, in all three, face recognition in a frame every 2 s inside each range: the box can show
   something else for a few seconds or minutes (a report, a weather map) and come back, which
   splits the range.
2. **Check the download.** The start and end of every range are cut the way users get them
   (`../scripts/download_*.py`) and compared with the broadcast's frames. The archives' streams
   do not always seek to the frame the broadcast had there: such a range's bounds are shifted by
   the frames it was off. A broadcast the archive no longer serves at the same height, or serves
   as another edit, is marked unavailable, with the reason.
3. **Keep** the audio of each range (for 4_labels), and a contact sheet of the ranges for
   3_verify's review.

Then the download is deleted. In 'update' mode only the broadcasts the release has not done are
done: ProTV and Digi24 days after the original run's, PrimaTV videos; and those of `REDO`, which
replace what earlier runs made of them, keeping the bounds that moved by at most a second.

Output, in `work/`: `segments.json`, for every broadcast its interpreter, its height, the ranges
[start, end) in frames of the broadcast (25 fps), the archive page, the shift of each range and
the segments as users download them, or why the archive no longer has it (None: no interpreter);
`audio/` and `sheets/`. The release's are in `frozen/segments.json`.

In [ ]:
import math
import shutil
import sys
import time
from collections import Counter
from functools import partial
from pathlib import Path
from types import SimpleNamespace

import cv2
import numpy as np
import requests

from common import CHANNELS, CROP, DECISIONS, FPS, FROZEN, INTERPRETERS, MODE, REPO, WORK, Classifier, FaceSearch, \
    IdentitySearch, Video, digi24_era, find_blocks, identity_reference, last_day, probe, read, save_audio, write

sys.path.insert(0, str(REPO / 'scripts'))   # the users' downloaders
import download_digi24  # noqa: E402
import download_prima   # noqa: E402
import download_protv   # noqa: E402

assert MODE != 'replay', "2_segment runs in 'live' or 'update' mode (common.MODE)"
COOKIES = 'cookies.txt'            # a logged-in YouTube session's cookies (PrimaTV; also needs the PO-token server, see README)
PAUSE, WAIT = 120, 30 * 60         # YouTube: seconds between two videos, and to wait while it refuses (PrimaTV below)
DOWNLOADS = WORK / 'downloads'     # one broadcast at a time
REDO = {'PrimaTV': set(), 'ProTV': set(), 'Digi24': set()}   # 'update' mode: broadcasts of the release to do again
OUT = WORK / 'segments.json'
broadcasts = read(WORK / 'broadcasts.json')
decisions = read(DECISIONS)
segments = read(OUT) if OUT.exists() else {'PrimaTV': {}, 'ProTV': {}, 'Digi24': {}}

if MODE == 'update':     # what the release has done
    before = read(FROZEN / 'segments.json')
    released = {c: set(before[c]) for c in segments}
    since = {c: last_day(c) for c in ('ProTV', 'Digi24')}


def to_do(channel, key):
    """Whether this run still has to do the broadcast."""
    if key in segments[channel]:
        return False
    if MODE == 'update':
        return key in REDO[channel] or (key not in released[channel]
                                         and (channel == 'PrimaTV' or key.split('/')[0] > since[channel]))
    return True


def settle(channel, key, ranges, within=FPS):
    """A broadcast done again keeps each bound the release had where the new one is within `within` frames of it: two
    searches can place a transition a frame or two apart, and the release's segments (and features) stay as they were."""
    old = ((before[channel].get(key) or {}).get('ranges') or []) if MODE == 'update' else []
    near = lambda bound, bounds: next((b for b in bounds if abs(b - bound) <= within), bound)
    return [[near(s, [b for b, _ in old]), near(e, [b for _, b in old])] for s, e in ranges]


def similarity(a, b):
    """Normalized cross-correlation of two images' gray levels (1: same picture). A flat picture (a fade to one
    colour) has nothing to correlate: two of them are as similar as their gray levels are close."""
    a, b = (np.asarray(x, np.float64).mean(-1).ravel() for x in (a, b))
    if min(a.std(), b.std()) < 2:
        return 1 - abs(a.mean() - b.mean()) / 255 if max(a.std(), b.std()) < 2 else 0.0
    a, b = a - a.mean(), b - b.mean()
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b)))


def shift_of(grab, frame, start, end, search=3):
    """The shift (in frames, -search..search) that best lines up the users' cut with the broadcast's
    first and last 3 frames of [start, end) (`frame(j)`: frame j, cropped as released), preferring
    the smallest of equally good ones; None if no shift makes them look alike (mean correlation
    under 0.9). The cut is re-encoded, so even the right frame only reaches about 0.98, and a static
    picture's neighbours as much: the start, usually a transition, decides."""
    tmp, scores = WORK / 'piece.mp4', {}
    for first in (start, end - 3):
        lo = max(0, first - search)
        grab(lo, first + 3 + search, tmp)
        cut = Video(tmp)
        got = [cut[i] for i in range(cut.frames)]
        for d in range(-search, search + 1):
            for j in range(first, first + 3):
                if 0 <= j + d - lo < len(got):
                    scores.setdefault(d, []).append(similarity(frame(j), got[j + d - lo]))
    mean = {d: float(np.mean(v)) for d, v in scores.items() if len(v) == 6}
    best = max(mean.values())
    if best < 0.9:
        return None
    return min((d for d, m in mean.items() if m >= best - 0.002), key=abs)


def check(sources, video, crop, ranges):
    """Where the users' downloads of the ranges come from: {'source', 'shift'} for the first archive video
    (url, grab) whose cuts line up with every range, or {'unavailable': reason}. grab(start, end,
    dest) cuts frames [start, end) of that video into dest as the users' downloader does."""
    frame = lambda j: video[j][crop['y']:crop['y'] + crop['h'], crop['x']:crop['x'] + crop['w']]
    try:
        for url, grab in sources:
            shifts = [shift_of(grab, frame, s, e) for s, e in ranges]
            if None not in shifts:
                return {'source': url, 'shift': shifts}
        raise RuntimeError('no video of the archive matches the broadcast')
    except Exception as error:
        return {'unavailable': str(error)[:300]}


def contact_sheet(channel, key, video, ranges):
    """work/sheets/<channel>/<key>.jpg, for 3_verify's review: frames 2 and 1 s before, at and 1 and 2 s after each range's
    start and last frame, then 6 frames across each range; for a broadcast done again, 6 frames across each stretch that
    it gained or lost. Each frame is shown whole and in the interpreter box."""
    c = CROP['Digi24'][digi24_era(key)] if channel == 'Digi24' else CROP[channel][video.height]
    old = ((before[channel].get(key) or {}).get('ranges') or []) if MODE == 'update' else []

    def tile(i, label):
        i = min(max(i, 0), video.frames - 2)
        f = cv2.cvtColor(video[i], cv2.COLOR_RGB2BGR)
        box = cv2.resize(f[c['y']:c['y'] + c['h'], c['x']:c['x'] + c['w']], (int(180 * c['w'] / c['h']), 180))
        t = np.concatenate([cv2.resize(f, (320, 180)), box], 1)
        cv2.putText(t, f'{label} {i / FPS:.1f}s', (4, 16), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (0, 255, 255), 1)
        return t

    def mask(rs):
        m = np.zeros(video.frames + 1, bool)
        for a, b in rs:
            m[a:b] = True
        return m
    rows = []
    for k, (a, b) in enumerate(ranges):
        rows.append([tile(a + d, f'{k} start{d:+d}') for d in (-2 * FPS, -FPS, 0, FPS, 2 * FPS)])
        rows.append([tile(b - 1 + d, f'{k} last{d:+d}') for d in (-2 * FPS, -FPS, 0, FPS, 2 * FPS)])
        rows.append([tile(a + (b - a) * q // 7, f'{k} in') for q in range(1, 7)])
    for name, changed in (('gained', mask(ranges) & ~mask(old)), ('lost', mask(old) & ~mask(ranges))) if old else ():
        edges = np.flatnonzero(np.diff(np.concatenate([[0], changed.astype(int), [0]])))
        rows += [[tile(a + (b - a) * q // 7, name) for q in range(1, 7)] for a, b in zip(edges[::2], edges[1::2])]
    width = max(sum(t.shape[1] for t in r) for r in rows)
    path = WORK / 'sheets' / channel / f'{key}.jpg'
    path.parent.mkdir(parents=True, exist_ok=True)
    cv2.imwrite(str(path), np.concatenate([np.pad(np.concatenate(r, 1), ((0, 2), (0, width - sum(t.shape[1] for t in r)),
                                                                       (0, 0))) for r in rows]))


def unrecognized(channel, key, video_path, spans=None):
    """Record a broadcast in which an interpreter was expected (PrimaTV: by 1_find; ProTV: every episode; Digi24: by
    the classifier) but no known face was found, with a contact sheet of `spans` (or the whole video), for 3_verify:
    it may show an interpreter `known_people/` does not have yet."""
    path = WORK / 'unrecognized.json'
    found = read(path) if path.exists() else {}
    found.setdefault(channel, [])
    if key not in found[channel]:
        found[channel].append(key)
    video = Video(video_path)
    contact_sheet(channel, key, video, spans or [[0, video.frames - 1]])
    write(path, found)


def keep(channel, key, video_path, interpreter, ranges, where=None, audio=None, **notes):
    """Record a broadcast (ranges None: no interpreter found) and save the audio of each range; a broadcast
    done again keeps the transcripts of the segments it has with the same frames, whose audio is not saved."""
    shutil.rmtree(WORK / 'audio' / channel / key, ignore_errors=True)    # of an interrupted attempt
    if ranges is None:
        segments[channel][key] = None
    else:
        earlier = (before[channel].get(key) or {}) if MODE == 'update' else {}
        old = (earlier.get('ranges') or []) if earlier.get('interpreter') else []   # none: never transcribed
        for i, (start, end) in enumerate(ranges):
            if i < len(old) and old[i] == [start, end]:
                continue
            save_audio(video_path, start, end, WORK / 'audio' / channel / key / f'{i}.flac', audio)
        record = {'interpreter': interpreter, 'source_height': Video(video_path).height, 'ranges': ranges, **where, **notes}
        if 'shift' in record:                     # the segments as users download them
            record['segments'] = [{'index': i, 'start': s + d, 'end': e + d}
                                  for i, ((s, e), d) in enumerate(zip(ranges, record['shift']))]
        segments[channel][key] = record
        contact_sheet(channel, key, Video(video_path), ranges)
    write(OUT, segments)


def download_digi24_video(src):
    """A Digi24 journal's video file -> DOWNLOADS/digi24.mp4."""
    DOWNLOADS.mkdir(parents=True, exist_ok=True)
    path = DOWNLOADS / 'digi24.mp4'
    headers = {'User-Agent': download_digi24.USER_AGENT, 'Referer': src, 'Accept-Encoding': 'identity',
               'Accept': 'video/webm,video/ogg,video/*;q=0.9,application/ogg;q=0.7,audio/*;q=0.6,*/*;q=0.5'}
    with requests.get(src, stream=True, timeout=60, headers=headers) as r:
        r.raise_for_status()
        with open(path, 'wb') as f:
            for chunk in r.iter_content(8 << 20):
                f.write(chunk)
    return path

## Face search

`common.find_blocks` with dlib's face recognition against one picture of each of the channel's
interpreters (`known_people/`, `common.INTERPRETERS`); Digi24's faces are too small and blurred
for that, and are matched with its identity check instead (below). PrimaTV and ProTV first look at a frame
every 30 s of the video: each run of them that show a known interpreter is a range, whose first
and last frame a binary search finds between the frames around it, refined frame by frame
(Digi24's ranges come from its classifier instead). Then, in all three, a frame every 2 s inside
each range, the same way, which splits it where the interpreter leaves for a few seconds (a
report, a transition). A frame counts as showing an interpreter if a face in the box in it or in
the frames 0.5 and 1 s after it is one. Ranges less than 2 s apart are joined, ranges under 10 s
dropped, and the interpreter recognized in most of the frames looked at labels the broadcast.
Frames are looked at on all CPUs.

`FACE_BOX` is where the face is looked for, by frame height: the notebooks' interpreter box
(numpy slicing clips it at the frame's edge); Digi24's is the released crop (`common.CROP`).

In [ ]:
FACE_BOX = {'PrimaTV': {1080: (1522, 720, 383, 411), 720: (1015, 480, 255, 274)},
            'ProTV': {720: (952, 347, 300, 322), 432: (571, 208, 180, 193)}}


def find_ranges(channel, video_path, box, frames, first=None, search=None):
    """(interpreter, [[start, end), ...]) of the frames [0, frames) of a broadcast showing one of the channel's
    interpreters in `box` (x, y, w, h): the ranges `first` (Digi24's), or those of a frame every 30 s, then checked
    every 2 s inside, by `search` (a common.FaceSearch by default); (None, []) if none."""
    search = search or FaceSearch(video_path, INTERPRETERS[channel], box)
    try:
        if first is None:
            first = [[f, l + 1] for f, l, _ in find_blocks(search, frames, stride=30 * FPS)]
        blocks = [inner for s, e in first for inner in find_blocks(search, e, start=s, stride=2 * FPS)]
    finally:
        search.close()
    seconds = sum((names for _, _, names in blocks), Counter())
    return (seconds.most_common(1)[0][0] if blocks else None), [[f, l + 1] for f, l, _ in blocks]

### PrimaTV

The videos with an interpreter: recognized by 1_find, or checked by eye (`decisions.json`,
which overrides the recognition). Each is downloaded as users get it: the video stream at the
highest height there is a crop for (1080p, else 720p), and its Romanian audio (YouTube also offers
automatic dubs of some videos), as opus: YouTube's AAC tracks start with about 35 ms of encoder
priming that their files do not signal, which would put the audio that much late; the opus tracks
signal theirs. Both come from one request for the video's formats, and the requests are at least
`PAUSE` seconds apart: YouTube stops serving a client that asks for a few dozen videos in a row.
When a download fails and a video YouTube served before fails too, it is refusing: the search
waits `WAIT` seconds and asks again. The face search ignores the video's last frame.

In [ ]:
from yt_dlp import YoutubeDL
from yt_dlp.utils import DownloadError

VIDEO = '/'.join(f'bv[height={h}][protocol=https]/bv[height={h}]' for h in (1080, 720))   # download_prima's, by height
AUDIO = 'ba[language=ro][acodec=opus]/ba[format_note*=original][acodec=opus]/ba[acodec=opus]'
youtube = {'format': f'({VIDEO}),({AUDIO})', 'outtmpl': str(DOWNLOADS / '%(id)s.%(format_id)s.%(ext)s'),
           'quiet': True, 'noprogress': True, 'retries': 10, 'cookiefile': COOKIES, 'js_runtimes': {'node': {}},
           'remote_components': ['ejs:github']}
checked, recognized = decisions['PrimaTV']['thumbnails'], broadcasts['PrimaTV']['recognized']


def served(url):
    """Whether YouTube answers for a video."""
    try:
        with YoutubeDL({**youtube, 'format': None}) as ydl:
            return bool(ydl.extract_info(url, download=False, process=False))
    except DownloadError:
        return False


served_before, asked = None, 0.0
for vid in broadcasts['PrimaTV']['videos']:
    if not to_do('PrimaTV', vid) or not (vid in REDO['PrimaTV'] or checked.get(vid, bool(recognized.get(vid)))):
        continue
    url, got = f'https://www.youtube.com/watch?v={vid}', None
    while got is None:
        time.sleep(max(0.0, asked + PAUSE - time.time()))
        try:
            with YoutubeDL(youtube) as ydl:
                got = ydl.extract_info(url)['requested_downloads']
        except DownloadError as error:
            if served_before and not served(served_before):
                print(vid, 'YouTube refuses; waiting', flush=True)
                time.sleep(WAIT)
                continue
            print(vid, 'not downloaded:', error)
            break
        finally:
            asked = time.time()
    if got is None:
        continue
    served_before = url
    (video_path,), (audio_path,) = ([Path(d['filepath']) for d in got if (d.get('vcodec') != 'none') == v] for v in (1, 0))
    video = Video(video_path)
    interpreter, ranges = find_ranges('PrimaTV', video_path, FACE_BOX['PrimaTV'][video.height], video.frames - 1)
    if ranges:
        ranges = settle('PrimaTV', vid, ranges)
        crop, t0 = CROP['PrimaTV'][video.height], download_prima.stream_start_time(video_path)
        grab = lambda s, e, dest: download_prima.grab_segment(video_path, t0, {'start': s, 'end': e}, crop, dest)
        keep('PrimaTV', vid, video_path, interpreter, ranges, check([(url, grab)], video, crop, ranges), audio=audio_path)
    else:
        unrecognized('PrimaTV', vid, video_path)
        keep('PrimaTV', vid, video_path, None, None)
    video_path.unlink()
    audio_path.unlink()

### ProTV

Up to 2024-02-29 the broadcasts come from protv.ro, at the highest height up to 720p; from
2024-03-01 from vid2, at 432p, as in the original run. While vid2 serves no files they come
from protv.ro's 432p version, which has the same frames (the original run found 249 of its vid2
broadcasts there unchanged). vid2 is downloaded at about 1 MB/s, as the original run did.

Users cut their segments out of protv.ro's stream: for a protv.ro broadcast, that of its page;
for a vid2 file, that of the protv.ro episode of the day it was stored, or of a day around it.
Only a few frames around each end are fetched for the check.

In [ ]:
from datetime import date, timedelta

VID2_FROM = '2024-03-01'


def download_protv_broadcast(route, url, height):
    """A ProTV broadcast -> DOWNLOADS/protv.(mp4|mkv)"""
    DOWNLOADS.mkdir(parents=True, exist_ok=True)
    if route == 'vid2':
        path = DOWNLOADS / 'protv.mp4'
        with requests.get(url, stream=True, timeout=60) as r, open(path, 'wb') as f:
            r.raise_for_status()
            for chunk in r.iter_content(1 << 20):
                f.write(chunk)
                time.sleep(1)
        return path
    form = f'bv[height<={height}]+ba/b[height<={height}]/b' if height == 720 else f'bv[height={height}]+ba'
    with YoutubeDL({'format': form, 'merge_output_format': 'mkv', 'quiet': True,
                    'outtmpl': str(DOWNLOADS / 'protv.%(ext)s'), 'retries': 10}) as ydl:
        ydl.download([download_protv.resolve_manifest_url(requests.Session(), url)])
    return DOWNLOADS / 'protv.mkv'


def protv_sources(key, height, crop):
    """(page, grab) for each protv.ro stream that may hold the broadcast, at its height."""
    day, name = key.split('/')
    if name.endswith('-2'):       # a vid2 file, stored on or about the day it aired
        near = [(date.fromisoformat(day) + timedelta(n)).isoformat() for n in (0, -1, 1, -2, 2)]
        pages = [u for x in near for u in pages_by_day.get(x, [])]
    else:
        pages = [u for u in pages_by_day.get(day, []) if episode(u) == name]
    for url in pages:
        try:
            source = download_protv.resolve_manifest_url(requests.Session(), url)
            stream = download_protv.pick_video_stream(source, height)
        except RuntimeError:
            continue
        t0 = download_protv.stream_start_time(source, stream)
        yield url, partial(lambda src, st, t, s, e, dest: download_protv.grab_segment(
            src, st, t, {'start': s, 'end': e}, crop, dest), source, stream, t0)


episode = lambda url: url.rstrip('/').rsplit('/', 1)[1].split('-')[0]
pages_by_day, files = broadcasts['ProTV']['protv.ro'], broadcasts['ProTV']['vid2']
jobs = [(f'{day}/{episode(u)}', 'protv.ro', u, 720) for day, us in pages_by_day.items() if day < VID2_FROM for u in us]
if files:
    jobs += [(f"{day}/{u.rsplit('/', 1)[1][:-4]}", 'vid2', u, 432) for day, us in files.items() for u in us]
else:
    jobs += [(f'{day}/{episode(u)}', 'protv.ro', u, 432) for day, us in pages_by_day.items() if day >= VID2_FROM for u in us]

for key, route, url, height in jobs:
    if not to_do('ProTV', key):
        continue
    try:
        video_path = download_protv_broadcast(route, url, height)
    except Exception as error:
        print(key, 'not downloaded:', error)
        continue
    video = Video(video_path)
    interpreter, ranges = find_ranges('ProTV', video_path, FACE_BOX['ProTV'][video.height], video.frames)
    if ranges:
        ranges, crop = settle('ProTV', key, ranges), CROP['ProTV'][video.height]
        keep('ProTV', key, video_path, interpreter, ranges, check(protv_sources(key, video.height, crop), video, crop, ranges))
    else:
        unrecognized('ProTV', key, video_path)
        keep('ProTV', key, video_path, None, None)
    video_path.unlink()

## Digi24

### Which days have an interpreter

A pose check on the interpreter box: exactly one person, facing the camera (nose, eyes and
shoulders seen, shoulders level within 20%), both elbows and wrists and both hips seen
(RTMW whole-body pose, confidence 0.5). A day has an interpreter if more than half of the
frames every 4 s from minute 5 to minute 15 pass it.

In [ ]:
import torch
from rtmlib import Wholebody

pose = Wholebody(mode='lightweight', backend='onnxruntime', device='cuda' if torch.cuda.is_available() else 'cpu')
COCO = {'nose': 0, 'left_eye': 1, 'right_eye': 2, 'left_shoulder': 5, 'right_shoulder': 6, 'left_elbow': 7,
        'right_elbow': 8, 'left_wrist': 9, 'right_wrist': 10, 'left_hip': 11, 'right_hip': 12}
digi24 = decisions['Digi24']


def looks_like_interpreter(image):
    keypoints, scores = pose(image)
    if keypoints.shape[0] != 1:
        return False
    k, s = keypoints[0], scores[0]
    seen = lambda *parts: all(s[COCO[p]] >= 0.5 for p in parts)
    y_left, y_right = k[COCO['left_shoulder'], 1], k[COCO['right_shoulder'], 1]
    return (seen('nose', 'left_eye', 'right_eye', 'left_shoulder', 'right_shoulder')
            and abs(y_left - y_right) <= 0.2 * (y_left + y_right)
            and seen('left_elbow', 'right_elbow', 'left_wrist', 'right_wrist', 'left_hip', 'right_hip'))


def pose_box(frame, day):
    """The interpreter box of the pose check (fractions of the 1280x720 frame)."""
    if digi24_era(day) == 'from 2023-12':
        return frame[int(0.400 * 720):int(0.705 * 720), int(0.755 * 1280):int(0.950 * 1280)]
    return frame[int(0.660 * 720):int(0.945 * 720), int(0.800 * 1280):int(0.945 * 1280)]


def has_interpreter(video_path, day):
    cap = cv2.VideoCapture(str(video_path))
    total, checks = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)), []
    for i in range(5 * 60 * FPS, min(15 * 60 * FPS, total - 1) + 1, 4 * FPS):
        cap.set(cv2.CAP_PROP_POS_FRAMES, i)
        ok, frame = cap.read()
        if not ok:
            break
        checks.append(looks_like_interpreter(pose_box(frame, day)))
    return bool(checks) and sum(checks) / len(checks) > 0.5


def page(day):
    """The day's journal page: the one chosen by hand if the day has several."""
    return digi24['pages'].get(day, broadcasts['Digi24']['listing'][day][0])


def download_digi24_day(day):
    """The day's journal (720p) -> DOWNLOADS/digi24.mp4."""
    pages = broadcasts['Digi24']['listing'][day]
    if len(pages) > 1 and day not in digi24['pages']:
        print(day, 'has', len(pages), 'pages; taking the first: add the right one to decisions.json (Digi24, pages)')
    return download_digi24_video(broadcasts['Digi24']['video_src'][page(day)])

### The classifier

Trained on the days with an interpreter: 40 frames of each, spread evenly, labelled by the pose
check above and padded to 250x219. The head of an ImageNet MobileNetV3-Small is trained
(backbone frozen), the minority class augmented and the classes weighted, until 99.5% of the
training frames are right. The original run's weights are `frozen/models/digi24_classifier.pt`;
set `TRAIN = True` to train new ones (this needs every day's download once more).

In [ ]:
TRAIN = False
WEIGHTS = FROZEN / 'models' / 'digi24_classifier.pt'
days = [d for d in broadcasts['Digi24']['listing'] if to_do('Digi24', d)]

if TRAIN:
    import itertools
    from PIL import Image
    from torch import nn
    from torch.utils.data import ConcatDataset, DataLoader, Dataset
    from torchvision import models, transforms as T

    def pad(frame, w=250, h=219):
        top, left = (h - frame.shape[0]) // 2, (w - frame.shape[1]) // 2
        return np.pad(frame, ((top, h - frame.shape[0] - top), (left, w - frame.shape[1] - left), (0, 0)))

    with_interpreter, without = [], []
    for day in days:
        path = download_digi24_day(day)
        if has_interpreter(path, day):
            cap = cv2.VideoCapture(str(path))
            total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
            for i in range(0, total, math.ceil(total / 40)):
                cap.set(cv2.CAP_PROP_POS_FRAMES, i)
                ok, frame = cap.read()
                if not ok:
                    break
                box = pose_box(frame, day)
                (with_interpreter if looks_like_interpreter(box) else without).append(pad(box))
        path.unlink()

    weights = models.MobileNet_V3_Small_Weights.DEFAULT
    mean, std = weights.meta.get('mean', (0.485, 0.456, 0.406)), weights.meta.get('std', (0.229, 0.224, 0.225))

    class Frames(Dataset):
        def __init__(self, frames, label, augment):
            h, w = frames[0].shape[:2]
            extra = [T.ColorJitter(0.1, 0.1, 0.1, 0.05), T.RandomRotation(5),
                     T.RandomResizedCrop((h, w), scale=(0.9, 1.0))] if augment else []
            self.frames, self.label, self.tfm = frames, label, T.Compose(extra + [T.ToTensor(), T.Normalize(mean, std)])

        def __len__(self):
            return len(self.frames)

        def __getitem__(self, i):
            return self.tfm(Image.fromarray(self.frames[i][..., ::-1])), self.label

    minority = len(with_interpreter) < len(without)
    data = DataLoader(ConcatDataset([Frames(with_interpreter, 1, minority), Frames(without, 0, not minority)]),
                      batch_size=512, shuffle=True, num_workers=4, pin_memory=True)
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    model = models.mobilenet_v3_small(weights=weights)
    model.classifier[3] = nn.Linear(model.classifier[3].in_features, 2)
    for p in itertools.chain(model.features.parameters(), model.avgpool.parameters()):
        p.requires_grad = False
    model.to(device)
    counts = torch.tensor([len(without), len(with_interpreter)], dtype=torch.float, device=device)
    loss_fn = nn.CrossEntropyLoss(weight=counts.sum() / (2 * counts))
    optimizer = torch.optim.AdamW(model.classifier.parameters(), lr=1e-3)
    best, patience = 0.0, 3
    for epoch in range(6):
        model.train()
        for x, y in data:
            loss = loss_fn(model(x.to(device)), y.to(device))
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            optimizer.step()
        model.eval()
        with torch.no_grad():
            right = sum((model(x.to(device)).argmax(1).cpu() == y).sum().item() for x, y in data)
        accuracy = right / len(data.dataset)
        print(f'epoch {epoch + 1}: {accuracy:.4f}')
        best, patience = (accuracy, 3) if accuracy > best else (best, patience - 1)
        if patience == 0 or best >= 0.995:
            break
    WEIGHTS = WORK / 'digi24_classifier.pt'
    torch.save(model.state_dict(), WEIGHTS)

### The ranges

The classifier sees every frame of the box. A range opens once the interpreter has been seen
for 2 s and closes once they have been gone for 2 s; ranges under 10 s are dropped. The end
kept is the last frame seen, as an exclusive end: the original run cut every range up to, not
including, that frame, and the released bounds keep it so.

In [ ]:
CLASSIFIER_BOX = {'until 2023-11': (1018, 1210, 474, 680), 'from 2023-12': (966, 1216, 287, 510)}  # x0, x1, y0, y1
classify = Classifier(WEIGHTS, pad_to=(250, 223))


def digi24_ranges(video_path, day):
    x0, x1, y0, y1 = CLASSIFIER_BOX[digi24_era(day)]
    ranges, start, gone, maybe = [], -1, -1, -1
    index = -1
    for batch in Video(video_path).read():
        for seen in classify([f[y0:y1, x0:x1] for f in batch]):
            index += 1
            if seen:
                if start >= 0:
                    gone = -1
                else:
                    maybe = index if maybe < 0 else maybe
                    if index - maybe >= 2 * FPS:
                        start, gone = maybe, -1
                continue
            maybe = -1
            if start >= 0 and gone < 0:
                gone = index
            if gone >= 0 and index - gone >= 2 * FPS:
                if gone - 1 - start >= 10 * FPS:
                    ranges.append([start, gone - 1])
                start, gone = -1, -1
    if start >= 0 and index - start >= 10 * FPS:
        ranges.append([start, index])
    return ranges

### Who is in the box

Digi24's face check: the face in the box (the released crop) is matched against reference frames
of each interpreter, 20 evenly spaced frames of each of the hand-checked segments in
`decisions.json` (Digi24, `reference_segments`), cut as users get them. FaceNet (MTCNN,
InceptionResnetV1 trained on VGGFace2) gives the distance to an interpreter as the mean of the 5
nearest reference frames. A frame is a hit if the nearest interpreter is the day's interpreter
(the most frequent nearest one), at distance 1.0 or less and 0.05 closer than the next; a frame
that is not is checked again with dlib (face_recognition: distance 0.55 or less, 0.03 closer than
the next). On 32 hand-checked segments this missed 0.3% of the right interpreter's frames and
accepted no other interpreter; dlib against one picture, as for PrimaTV and ProTV, misses the
interpreter in blurred frames (`common.IdentitySearch`).

In [ ]:
def reference_frames():
    """The reference frames of each reference segment (RGB), by interpreter, fetched once into work/reference.npz."""
    path = WORK / 'reference.npz'
    if not path.exists():
        frames, tmp = {}, WORK / 'piece.mp4'
        listing = {**read(FROZEN / 'broadcasts.json')['Digi24']['listing'], **broadcasts['Digi24']['listing']}
        for name, ref in digi24['reference_segments'].items():
            day = name.split('/')[0]              # a day of the original run's
            src = download_digi24.resolve_video_src(requests.Session(), digi24['pages'].get(day, listing[day][0]), 720)
            t0, crop, n = download_digi24.stream_start_time(src), CROP['Digi24'][digi24_era(day)], ref['end'] - ref['start']
            for k in range(20):
                j = ref['start'] + int((k + 0.5) * n / 20)
                download_digi24.grab_segment(src, t0, {'start': j, 'end': j + 1}, crop, tmp)
                frames[f"{ref['interpreter']}/{name}/{k}"] = Video(tmp)[0]
        np.savez_compressed(path, **frames)
    reference = {}
    for key, frame in np.load(path).items():
        interpreter, day, i, _ = key.split('/')
        reference.setdefault(interpreter, {}).setdefault(f'{day}/{i}', []).append(frame)
    return {interpreter: list(by_segment.values()) for interpreter, by_segment in reference.items()}


def who_is_in_the_box(video_path, crop, first):
    """(interpreter, ranges) of a Digi24 day: the classifier's ranges `first`, checked inside as above."""
    box = (crop['x'], crop['y'], crop['w'], crop['h'])
    search = IdentitySearch(video_path, box, reference)
    nearest = Counter(n for n in search.nearest([i for s, e in first for i in range(s, e, 2 * FPS)]) if n)
    if not nearest:
        return None, []
    search.expected = nearest.most_common(1)[0][0]
    return find_ranges('Digi24', video_path, box, Video(video_path).frames, first, search)


if days:
    reference = identity_reference(reference_frames())

### Each day

In [ ]:
for day in days:
    video_path = download_digi24_day(day)
    if not has_interpreter(video_path, day):
        keep('Digi24', day, video_path, None, None)
    else:
        crop, first = CROP['Digi24'][digi24_era(day)], digi24_ranges(video_path, day)
        interpreter, ranges = who_is_in_the_box(video_path, crop, first)
        if not ranges:
            if first:
                unrecognized('Digi24', day, video_path, first)
            keep('Digi24', day, video_path, None, None)
            video_path.unlink()
            continue
        ranges = settle('Digi24', day, ranges)
        t0 = download_digi24.stream_start_time(str(video_path))
        grab = lambda s, e, dest: download_digi24.grab_segment(str(video_path), t0, {'start': s, 'end': e}, crop, dest)
        where = check([(page(day), grab)], Video(video_path), crop, ranges)
        keep('Digi24', day, video_path, interpreter, ranges, where, video_src=broadcasts['Digi24']['video_src'][page(day)])
    video_path.unlink()

## Ranges searched before the check inside them

The release's ranges found before 2_segment checked inside them (in `frozen/segments.json`, those
without `segments`) were searched on broadcasts that have since been deleted. The check runs on
each range as users download it instead: their file in `../mp4s/` if it holds the range whole,
otherwise cut with the users' downloaders into `work/inside_check/` (each PrimaTV video downloaded
once, `PAUSE` s apart). Digi24's reference frames come from the users' files of the reference
segments. Bounds within 1 s of the range's own or of the release's segments stay where they are.
The result of each range is kept in `work/inside_check_<channel>.json`, so the cell resumes.

In [ ]:
import os
import subprocess
from multiprocessing import Pool

MANIFEST = {'PrimaTV': 'prima', 'ProTV': 'protv', 'Digi24': 'digi24'}
FETCHED = WORK / 'inside_check'
DOWNLOADERS = {'PrimaTV': download_prima, 'ProTV': download_protv, 'Digi24': download_digi24}
reference_downloads = None


def unchecked(channel):
    """{<name>/<i>: (key, name, i, [start, end) as users download it, crop, source, height, interpreter)} of the
    ranges of the release's broadcasts that were not checked inside."""
    out = {}
    for key, record in before[channel].items():
        if not record or not record['interpreter'] or 'unavailable' in record or 'segments' in record \
                or key in decisions[channel].get('dropped_days', {}):
            continue
        name = key.split('/')[0] if channel == 'ProTV' else key
        crop = CROP['Digi24'][digi24_era(name)] if channel == 'Digi24' else CROP[channel][record['source_height']]
        for i, ((s, e), d) in enumerate(zip(record['ranges'], record['shift'])):
            out[f'{name}/{i}'] = (key, name, i, (s + d, e + d), crop, record.get('video_src') or record['source'],
                                  record['source_height'], record['interpreter'])
    return out


def range_video(channel, key, name, index, span, crop, source, height, interpreter):
    """The range as users download it: their file if it holds it whole, else one cut from the archive."""
    local = DOWNLOADERS[channel].segment_path({'id': name, 'date': name}, index, REPO / 'mp4s' / channel)
    if local.exists() and Video(local).frames == span[1] - span[0]:
        return local
    dest = FETCHED / channel / f'{name}_{index}.mp4'
    if not dest.exists():
        dest.parent.mkdir(parents=True, exist_ok=True)
        segment = {'start': span[0], 'end': span[1]}
        if channel == 'Digi24':
            download_digi24.grab_segment(source, download_digi24.stream_start_time(source), segment, crop, dest)
        elif channel == 'ProTV':
            stream_url = download_protv.resolve_manifest_url(requests.Session(), source)
            stream = download_protv.pick_video_stream(stream_url, height)
            download_protv.grab_segment(stream_url, stream, download_protv.stream_start_time(stream_url, stream),
                                        segment, crop, dest)
        else:
            raise RuntimeError('not downloaded')  # fetch_primatv() downloads them
    return dest


def fetch_primatv(todo):
    """Download each PrimaTV video with ranges the users' files do not hold once, cut them all and delete it."""
    users = SimpleNamespace(yt_dlp=str(Path(sys.executable).with_name('yt-dlp')), cookies=COOKIES, js_runtime='node',
                            remote_components=True)
    by_video = {}
    for key, name, index, span, crop, source, height, _ in todo.values():
        local = download_prima.segment_path({'id': name}, index, REPO / 'mp4s' / 'PrimaTV')
        dest = FETCHED / 'PrimaTV' / f'{name}_{index}.mp4'
        if not dest.exists() and not (local.exists() and Video(local).frames == span[1] - span[0]):
            by_video.setdefault((name, source, height), []).append((span, crop, dest))
    asked, served = 0.0, None
    for (name, source, height), wanted in by_video.items():
        while True:
            time.sleep(max(0.0, asked + PAUSE - time.time()))
            try:
                full = download_prima.download_stream(users, {'id': name, 'url': source, 'source_height': height}, FETCHED)
                break
            except RuntimeError as error:
                asked = time.time()
                if served and subprocess.run(download_prima.ytdlp_base(users) + ['--skip-download', '-q', served],
                                             capture_output=True).returncode != 0:
                    print('YouTube refuses; waiting', flush=True)
                    time.sleep(WAIT)
                    continue
                print(name, 'not downloaded:', str(error)[-200:], flush=True)
                full = None
                break
        asked = time.time()
        if full is None:
            continue
        served, t0 = source, download_prima.stream_start_time(full)
        for span, crop, dest in wanted:
            dest.parent.mkdir(parents=True, exist_ok=True)
            download_prima.grab_segment(full, t0, {'start': span[0], 'end': span[1]}, crop, dest)
        full.unlink()
        print(name, 'downloaded', flush=True)


def check_inside(job):
    """[[start, end), ...] of the range's frames showing the interpreter (from the range's first frame), None if that is
    all of it, or the error that kept it from being checked."""
    channel, args, bounds = job
    if args[3][1] - args[3][0] < 10 * FPS:   # too short to hold a segment (find_blocks' `shortest`): none, unseen
        return []
    try:
        mp4 = range_video(channel, *args)
    except Exception as error:
        return {'error': str(error)[:200]}
    video = Video(mp4)
    box = (0, 0, int(probe(mp4, 'width')['width']), video.height)
    search = IdentitySearch(mp4, box, reference_downloads, args[7]) if channel == 'Digi24' else \
        FaceSearch(mp4, INTERPRETERS[channel], box, processes=1)
    found = [[first, last + 1] for first, last, _ in find_blocks(search, video.frames, stride=2 * FPS)]
    keep_at = [b - args[3][0] for b in bounds] + [0, video.frames]   # the release's bounds first
    found = [[next((k for k in keep_at if abs(k - b) <= FPS), b) for b in part] for part in found]
    return None if found == [[0, video.frames]] else found


def digi24_reference_downloads():
    """identity_reference of 20 evenly spaced frames of each reference segment, from the users' file of it."""
    starts = {f"{it['date']}/{s['index']}": s['start']
              for it in read(REPO / 'manifests' / 'digi24_manifest.json')['items'] for s in it['segments']}
    frames = {}
    for name, ref in digi24['reference_segments'].items():
        day, index = name.split('/')
        video = Video(download_digi24.segment_path({'date': day}, int(index), REPO / 'mp4s' / 'Digi24'))
        at = [ref['start'] + int((k + 0.5) * (ref['end'] - ref['start']) / 20) - starts[name] for k in range(20)]
        frames.setdefault(ref['interpreter'], []).append([video[j] for j in at])
    return identity_reference(frames)


if MODE == 'update':
    for channel in CHANNELS:
        todo, done_path = unchecked(channel), WORK / f'inside_check_{channel}.json'
        if not todo:
            continue
        done = read(done_path) if done_path.exists() else {}
        todo = {k: v for k, v in todo.items() if k not in done or isinstance(done[k], dict)}
        bounds = {it.get('id') or it['date']: [b for s in it['segments'] for b in (s['start'], s['end'])]
                  for it in read(REPO / 'manifests' / f'{MANIFEST[channel]}_manifest.json')['items']}
        jobs = [(channel, v, [b for b in bounds.get(v[1], []) if v[3][0] <= b <= v[3][1]]) for v in todo.values()]
        if channel == 'PrimaTV':
            fetch_primatv(todo)
        def record(results):
            for n, (k, result) in enumerate(zip(todo, results), 1):
                done[k] = result
                if n % 20 == 0 or n == len(todo):
                    write(done_path, done)
        if channel == 'Digi24' and todo:      # FaceNet on the GPU: in this process
            reference_downloads = digi24_reference_downloads()
            record(map(check_inside, jobs))
        elif todo:
            with Pool(os.cpu_count()) as pool:
                record(pool.imap(check_inside, jobs))
        failed = {k: v['error'] for k, v in done.items() if isinstance(v, dict)}
        if failed:                            # left as they were found
            print(channel, len(failed), 'ranges could not be checked:', failed)
        for key, record in before[channel].items():  # the checked broadcasts, with their segments
            name = key.split('/')[0] if channel == 'ProTV' else key
            ranges = [k for k in done if k.rsplit('/', 1)[0] == name]
            if not ranges or key in segments[channel] or 'unavailable' in (record or {}):
                continue
            parts, n = [], len(record['ranges'])  # a range's parts after its first: numbered after the ranges
            for i, ((s, e), d) in enumerate(zip(record['ranges'], record['shift'])):
                result = done.get(f'{name}/{i}')
                for k, (a, b) in enumerate([[0, e - s]] if result is None or isinstance(result, dict) else result):
                    index, n = (i, n) if k == 0 else (n, n + 1)
                    parts.append({'index': index, 'start': s + d + a, 'end': s + d + b})
            segments[channel][key] = {**record, 'segments': sorted(parts, key=lambda p: p['start'])}
        write(OUT, segments)
        # ranges kept that were never transcribed (Digi24 segments a review by hand had dropped): their audio, from the
        # archive, for 4_labels
        transcribed = set(read(FROZEN / 'transcripts.json.zst')[channel])
        for key, record in before[channel].items():
            name = key.split('/')[0] if channel == 'ProTV' else key
            need = [i for i, (s, e) in enumerate((record or {}).get('ranges') or []) if f'{name}/{i}' in done
                    and done[f'{name}/{i}'] != [] and f'{key}/{i}' not in transcribed and key in segments[channel]]
            if not need:
                continue
            if channel != 'Digi24':
                print(channel, key, 'ranges', need, 'have no transcript: do the broadcast again (REDO)')
                continue
            video_path = download_digi24_video(record['video_src'])
            for i in need:
                save_audio(video_path, *record['ranges'][i], WORK / 'audio' / channel / key / f'{i}.flac')
            video_path.unlink()
            print(channel, key, 'ranges', need, 'had no transcript: audio saved')
        print(channel, len(done), 'ranges checked,', sum(isinstance(v, list) for v in done.values()), 'cut or dropped')